GenoSyntax

# GenoSyntax: A Framework for Deconstructing and Quantifying Genomic Architecture

[![Python Version](https://img.shields.io/badge/python-3.9+-blue.svg)](https://www.python.org/downloads/)

GenoSyntax is a bioinformatics framework designed to analyze the genomic architecture of related microbial genomes, particularly phages and bacteria. It moves beyond simple gene content analysis to deconstruct the physical layout of genomes into a stable, conserved **backbone** and dynamic, plastic **slots**. By systematically quantifying the properties of both, GenoSyntax provides a comprehensive and quantitative view of genomic rigidity and plasticity.

## Key Features

*   **Integrated Pipeline:** A single `run` command orchestrates the entire workflow, from raw annotation files to final mechanistic insights, managing all intermediate data automatically.
*   **Quantitative Architectural Metrics:** Introduces novel summary metrics for high-level comparison:
    *   **Genomic Rigidity Index (GRI):** A normalized score (0-1) quantifying the stability and conservation of the genomic backbone.
    *   **Genomic Plasticity Profile:** A suite of three scores that provide a multi-faceted view of accessory genome dynamics.
*   **Robust Anchor-Slot Framework:** Algorithmically identifies conserved "Anchor Modules" (the backbone) and partitions the remaining "Slot Space" where mobile elements reside.
*   **Dual-Tier Statistical Testing:** Employs a rigorous permutation test to identify significant insertion hotspots and coldspots at two resolutions: for all accessory genes and specifically for multi-gene cassettes.
*   **Mechanism Exploration Engine:** Prepares high-quality, analysis-ready input files for specialized downstream tools to investigate motifs (`MEME`), evolutionary dynamics (`MAFFT`), and functional module composition (`Cytoscape`).

## Conceptual Framework

GenoSyntax is built on the **Anchor-Slot Model**:

1.  **Pangenome Stratification:** The pangenome is first divided into `Strict Core`, `Soft Core`, and `Accessory` gene sets based on prevalence.
2.  **Anchor Modules:** The structural backbone is defined by **Anchor Modules**—contiguous blocks of `Strict Core` genes that are syntenically conserved across the cohort. The stability of each anchor is quantified by its Sequence Identity (**ASI**) and Gene Order Index (**AOI**).
3.  **Slot Space:** All regions not occupied by anchors constitute the **Slot Space**. This is where genomic plasticity occurs. Slots are defined between anchors (`Inter-Anchor`), within anchors (`Intra-Anchor`), and at the genome termini (`Upstream`/`Downstream`).
4.  **Accessory Modules (AMs):** The accessory genome is organized into **Accessory Modules** (AMs), which can be single genes or co-evolving, physically adjacent multi-gene cassettes.
5.  **Structural Plasticity Index (SPI):** The mobility of each AM is quantified by its **SPI**, which is the number of unique slot types it occupies across the cohort.

## Installation

GenoSyntax requires Conda for environment management and relies on a few key external bioinformatics tools.

**1. Prerequisites:**
*   [Conda](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html) (Miniconda or Anaconda)
*   Git

**2. Clone the Repository:**
```bash
git clone https://github.com/yunzhe-liu/GenoSynax.git
cd GenoSyntax
```

**3. Create and Activate Conda Environment:**
```bash
conda create -n genosyntax python=3.9
conda activate genosyntax
```

**4. Install Dependencies:**
Install Python packages via `pip` and the required external tools via `conda`.
```bash
# Install Python packages
pip install -r requirements.txt

# Install essential external tools
conda install -c bioconda mafft
```

## Input Data Requirements

GenoSyntax requires four primary inputs:

1.  **Pan-genome Matrix (`--panaroo_file`):** The `gene_presence_absence_roary.csv` file generated by Panaroo.
2.  **Annotation Directory (`--pharokka_dir`):** The main output directory from Pharokka. GenoSyntax expects this directory to contain a subdirectory for each genome, which in turn holds the `.gff`, `.faa`, and reoriented `.fna` files.
3.  **Raw Sequence Directory (`--fasta_dir`):** A directory containing the original, raw FASTA sequence files for each genome (one file per genome). This is used for the primary assembly quality control.
4.  **Protein Sequence Directory (`--faa_dir`):** A directory containing protein FASTA (`.faa`) files. For Pharokka users, this is the same as `--pharokka_dir`.

## Usage

### Recommended: The Integrated Pipeline (`run` command)

The `run` command is the most convenient way to use GenoSyntax. It automates the entire workflow and organizes all outputs neatly.

**Basic Usage (Core Analysis: Steps 1-3 + Summary):**
This is the default and most common use case. It performs architecture definition, slot analysis, and statistical testing, concluding with the generation of a comprehensive summary report.

```bash
python main.py run \
    --panaroo_file /path/to/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --fasta_dir /path/to/raw_genome_fastas/ \
    --output_dir ./My_GenoSyntax_Analysis/
```

**Full Pipeline with Mechanism Exploration (Steps 1-4):**
To run the complete pipeline, including the final mechanism analysis step, use the `--mechanisms` flag.

```bash
python main.py run \
    --panaroo_file /path/to/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --fasta_dir /path/to/raw_genome_fastas/ \
    --output_dir ./My_Full_Analysis/ \
    --mechanisms motif evo_dynamics adjacency_analysis extract_layouts \
    --until-step 4
```

**Key Pipeline Arguments:**

*   `--output_dir`: Path to the main output directory. All results will be organized into subdirectories within this location.
*   `--until-step [1|2|3|4]`: (Default: 3) Stop the pipeline after the specified step is complete.
*   `--mechanisms [motif|evo_dynamics|...]`: A space-separated list of analyses to run in Step 4. This step is skipped by default unless this flag is used or `--until-step 4` is specified.
*   `--dry_run`: A flag to check all inputs and pipeline logic without running any heavy computations.

You can also pass module-specific parameters (e.g., `--tolerance`, `--permutations`) directly to the `run` command. Use `python main.py run -h` for a full list of options.

### Advanced: Step-by-Step Execution

For custom workflows or debugging, you can run each module individually. Note that you are responsible for providing the correct paths to the outputs from the previous step.

**Step 1: `define_architecture`**
```bash
python main.py define_architecture \
    --panaroo_file /path/to/gene_presence_absence_roary.csv \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --fasta_dir /path/to/raw_fastas/ \
    --output_dir ./output/1_define_architecture/
```

**Step 2: `analyze_slots`**
```bash
python main.py analyze_slots \
    --architecture_file ./output/1_define_architecture/architecture_definition.json \
    --intermediate_dir_prev ./output/1_define_architecture/intermediate/ \
    --output_dir ./output/2_slot_analysis/
```

**Step 3: `run_stats`**
```bash
python main.py run_stats \
    --slot_analysis_dir ./output/2_slot_analysis/ \
    --output_dir ./output/3_statistical_test/
```

**Step 4: `explore_mechanisms`**
```bash
python main.py explore_mechanisms \
    --analyses motif evo_dynamics \
    --def_arch_dir ./output/1_define_architecture/ \
    --slot_analysis_dir ./output/2_slot_analysis/ \
    --stats_dir ./output/3_statistical_test/ \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --output_dir ./output/4_mechanism_analysis/
```

**Step 5 (Optional): `plot_layouts`**

After running the `extract_layouts` analysis in Step 4, you can generate a high-quality visualization of the genomic architectures.

```bash
python main.py plot_layouts \
    --input ./output/4_mechanism_analysis/reports/all_genome_layouts.csv \
    --output my_genome_comparison.png \
    --sort
```

## Output Structure

A successful run will create the following structure in your main output directory:

```
My_GenoSyntax_Analysis/
├── 1_define_architecture/
│   ├── architecture_definition.json  # The master blueprint of the genomic architecture
│   └── intermediate/                 # Pre-processed databases (gene order, etc.)
│
├── 2_slot_analysis/
│   ├── reports/
│   │   └── module_dynamics_report.csv  # Key report detailing SPI, frequency, and class of all AMs
│   └── intermediate/
│       ├── slot_database.json
│       └── accessory_module_definitions.json
│
├── 3_statistical_test/
│   └── reports/
│       ├── permutation_test_slot_significance.csv          # Hotspot report for ALL accessory genes
│       └── permutation_test_multigene_module_significance.csv  # Hotspot report for MULTI-GENE cassettes ONLY
│
├── 4_mechanism_analysis/
│   ├── motif_analysis_all_modules/
│   │   └── hot_frontend_boundaries.fasta   # FASTA files for motif discovery
│   ├── motif_analysis_multigene_modules/
│   │   └── hot_frontend_boundaries.fasta   # Cassette-specific FASTA files (recommended for integrase motifs)
│   ├── motif_analysis_high_spi_enriched/   # <--- ADDED THIS DIRECTORY
│   │   └── high_spi_freq_frontend_boundaries.fasta
│   ├── adjacency_module_analysis/
│   │   ├── adjacency_modules_summary.csv   # Summary of multi-gene modules
│   │   ├── adjacency_nodes.csv             # Network files for Cytoscape
│   │   └── adjacency_edges.csv
│   └── reports/
│       ├── all_genome_layouts.csv          # Physical map of anchors/slots for visualization
│       ├── all_genomes_cds_layout.csv      # Detailed map of all coding sequences
│       ├── spi_vs_ped_analysis.csv         # Report linking structural plasticity (SPI) and protein evolution (PED)
│       └── high_spi_slot_enrichment.csv    # Statistical report on slots preferred by mobile elements
│
└── comprehensive_summary_report.json     # Summary
```



## Citation

If you use GenoSyntax in your research, please cite:
*(Manuscript in preparation. Citation details will be available here upon publication.)*


requirements.txt 

In [ ]:
# -----------------------------------------------------------------
# requirements.txt for GenoSyntax v1.0
# -----------------------------------------------------------------
#
# To install all dependencies, run:
# pip install -r requirements.txt
#

# --- Core Data Science & Scientific Computing ---
# Used for data manipulation, numerical operations, and statistics.
pandas
numpy
scipy

# --- Bioinformatics Core ---
# Used for sequence parsing, alignment handling, and other bio-related tasks.
biopython

# --- Network Analysis ---
# Required for the community detection step in accessory module discovery.
networkx
python-louvain

# --- Statistical Analysis ---
# Used for multiple testing correction (e.g., FDR calculation).
statsmodels

# --- Utilities ---
# Provides progress bars for a better user experience in the command line.
tqdm

main.py

In [ ]:
# /GenoSynax_test/main.py

import argparse
import os
import sys
from argparse import Namespace

# --- Package Setup ---
project_root = os.path.dirname(os.path.abspath(__file__))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- Module Entrypoint Imports ---
try:
    from genosyntax.define_architecture import run as run_da
    from genosyntax.slot_analysis import run as run_sa
    from genosyntax.statistical_test import run as run_st
    from genosyntax.mechanism_analysis import run as run_ma
    from genosyntax import summarizer as run_summary
except ImportError as e:
    print(f"\\nFATAL: Could not import a necessary analysis module: {e}")
    sys.exit(1)

def run_pipeline(args):
    """Orchestrates the sequential execution of the GenoSyntax pipeline."""
    # (The run_pipeline function from the previous answer remains unchanged, it is correct)
    print("====== GenoSyntax v1.0: Initiating Integrated Pipeline Run ======\\n")

    main_output_dir = args.output_dir
    dir_step1 = os.path.join(main_output_dir, "1_define_architecture")
    dir_step2 = os.path.join(main_output_dir, "2_slot_analysis")
    dir_step3 = os.path.join(main_output_dir, "3_statistical_test")
    dir_step4 = os.path.join(main_output_dir, "4_mechanism_analysis")

    os.makedirs(dir_step1, exist_ok=True); os.makedirs(dir_step2, exist_ok=True)
    os.makedirs(dir_step3, exist_ok=True); os.makedirs(dir_step4, exist_ok=True)
    
    print("\n>>> EXECUTING STEP 1: define_architecture...")
    args_step1 = Namespace(**vars(args)); args_step1.output_dir = dir_step1
    if not run_da(args_step1):
        print("[ERROR] Pipeline halted due to failure in Step 1."); return
    if args.until_step == 1:
        print("\\n====== Pipeline finished after Step 1 as requested. ======"); return

    print("\n>>> EXECUTING STEP 2: analyze_slots...")
    args_step2 = Namespace(**vars(args))
    args_step2.architecture_file = os.path.join(dir_step1, "architecture_definition.json")
    args_step2.intermediate_dir_prev = os.path.join(dir_step1, "intermediate")
    args_step2.output_dir = dir_step2
    if not run_sa(args_step2):
        print("[ERROR] Pipeline halted due to failure in Step 2."); return
    if args.until_step == 2:
        print("\\n====== Pipeline finished after Step 2 as requested. ======"); return

    print("\n>>> EXECUTING STEP 3: run_stats...")
    args_step3 = Namespace(**vars(args))
    args_step3.slot_analysis_dir = dir_step2
    args_step3.output_dir = dir_step3
    if not run_st(args_step3):
        print("[ERROR] Pipeline halted due to failure in Step 3."); return

    print("\n>>> EXECUTING STEP 3.5: Generating Comprehensive Summary...")
    if not args.dry_run:
        if not run_summary.run(main_output_dir):
            print("[WARNING] Comprehensive summary generation failed, but pipeline will continue.")

    should_run_step_4 = (args.until_step == 4) or (args.mechanisms is not None)
    if not should_run_step_4:
        print("\\n====== Pipeline finished after Step 3 as requested. ======"); return

    print("\n>>> EXECUTING STEP 4: explore_mechanisms...")
    mechanisms_to_run = args.mechanisms if args.mechanisms else ['motif', 'evo_dynamics', 'adjacency_analysis', 'extract_layouts']
    
    args_step4 = Namespace(**vars(args))
    args_step4.analyses = mechanisms_to_run
    args_step4.def_arch_dir = dir_step1
    args_step4.slot_analysis_dir = dir_step2
    args_step4.stats_dir = dir_step3
    args_step4.output_dir = dir_step4
    
    if not run_ma(args_step4):
        print("[ERROR] Pipeline halted due to failure in Step 4."); return
        
    print("\\n====== Pipeline finished after Step 4. ======")


def main():
    """Main function to configure and parse command-line arguments."""
    parser = argparse.ArgumentParser(
        prog="genosyntax",
        description="GenoSyntax v1.0: A Framework for Genomic Architecture and Plasticity Analysis.",
        formatter_class=argparse.RawTextHelpFormatter
    )
    parser.add_argument('-v', '--version', action='version', version='%(prog)s 1.0')

    subparsers = parser.add_subparsers(dest='command', required=True, help="Available commands")

    # --- Parent parser for shared STEP 1 arguments ---
    da_args = argparse.ArgumentParser(add_help=False)
    da_args.add_argument('--panaroo_file', required=True, help='Path to Panaroo gene_presence_absence_roary.csv file.')
    da_args.add_argument('--pharokka_dir', required=True, help='Path to Pharokka output directory (contains .gff, .faa, etc.).')
    da_args.add_argument('--fasta_dir', required=True, help='Path to directory with raw genome FASTA files (for QC).')
    da_args.add_argument('--faa_dir', help='Path to directory with protein FASTA (.faa) files.')
    da_args.add_argument('--output_dir', required=True, help='Path to the output directory for this module/pipeline.')
    da_args.add_argument('--max_contigs', type=int, default=1, help='(Step 1) Max contigs for a genome to pass QC (default: 1).')
    da_args.add_argument('--orientation_consistency_threshold', type=float, default=0.9, help='(Step 1) Threshold for consistent genome orientation (default: 0.9).')
    da_args.add_argument('--tolerance', type=int, default=2, help='(Step 1) Max accessory genes allowed within an anchor (default: 2).')
    da_args.add_argument('--coverage_threshold', type=float, default=0.9, help='(Step 1) Min fraction of genomes an anchor must be in (default: 0.9).')
    da_args.add_argument('--min_length', type=int, default=2, help='(Step 1) Min number of core genes in an anchor (default: 2).')

    # ============================ START OF S1 MODIFICATION ============================
    # Add new arguments for core gene stratification thresholds to the shared parent parser.
    da_args.add_argument('--strict_core_threshold', type=float, default=1.0, help="(Step 1) Prevalence threshold for a gene to be 'strict core' (default: 1.0).")
    da_args.add_argument('--soft_core_threshold', type=float, default=0.98, help="(Step 1) Prevalence threshold for a gene to be 'soft core', also defines accessory boundary (default: 0.98).")
    # ============================= END OF S1 MODIFICATION =============================

    # --- Sub-parser for the 'run' command ---
    p_run = subparsers.add_parser('run', help='(Recommended) Run the entire GenoSyntax pipeline sequentially.', parents=[da_args])
    p_run.add_argument('--spi_resolution', choices=['low', 'high'], default='low', help="(Step 2) Resolution for SPI calculation (default: 'low').")
    p_run.add_argument('--permutations', type=int, default=10000, help="(Step 3) Number of permutations for statistical tests (default: 10000).")
    p_run.add_argument('--spi_enrichment_threshold', type=float, default=1.0, help="(Step 4, evo_dynamics) SPI value above which a module is 'high-SPI' (default: 1.0).")
    p_run.add_argument('--until-step', type=int, default=3, choices=[1, 2, 3, 4], help='Stop the pipeline after this step. Default is 3.')
    p_run.add_argument('--mechanisms', nargs='+', choices=['motif', 'evo_dynamics', 'adjacency_analysis', 'extract_layouts', 'extract_cds_layout'], help='(Optional) Select one or more mechanism analyses to run as Step 4.')
    p_run.add_argument('--dry_run', action='store_true', help='Check inputs and logic for all steps without running heavy computations.')

    # ============================ START OF S1 MODIFICATION ============================
    # Add new arguments for Accessory Module discovery to the 'run' command parser.
    p_run.add_argument('--phi_threshold', type=float, default=0.8, help="(Step 2) Phi coefficient threshold for gene co-occurrence (default: 0.8).")
    p_run.add_argument('--p_value_threshold', type=float, default=0.01, help="(Step 2) P-value threshold for gene co-occurrence (default: 0.01).")
    p_run.add_argument('--adjacency_distance', type=int, default=250, help="(Step 2) Max distance (bp) between adjacent genes in a module (default: 250).")
    p_run.add_argument('--support_threshold', type=float, default=0.8, help="(Step 2) Min fraction of genomes where a module must be physically adjacent (default: 0.8).")
    # ============================= END OF S1 MODIFICATION =============================

    # ==============================================================================
    # --------------- DEFINING SUBPARSERS FOR INDIVIDUAL MODULES ------------------
    # ==============================================================================
    p_da = subparsers.add_parser('define_architecture', help='(Step 1) Identify and characterize the core genomic architecture.', parents=[da_args])
    p_da.add_argument('--dry_run', action='store_true', help='Check inputs and logic without running heavy computations.')

    p_sa = subparsers.add_parser('analyze_slots', help='(Step 2) Partition slot space and analyze accessory module dynamics.')
    p_sa.add_argument('--architecture_file', required=True, help='Path to the architecture_definition.json from the define_architecture step.')
    p_sa.add_argument('--intermediate_dir_prev', required=True, help="Path to the 'intermediate' output directory from the define_architecture step.")
    p_sa.add_argument('--output_dir', required=True, help='Path to the output directory for this module.')
    p_sa.add_argument('--spi_resolution', choices=['low', 'high'], default='low', help="Resolution for SPI calculation (default: 'low').")
    p_sa.add_argument('--dry_run', action='store_true', help='Check inputs and logic without running heavy computations.')

    # ============================ START OF S1 MODIFICATION ============================
    # Add the same Accessory Module arguments to the standalone 'analyze_slots' command.
    p_sa.add_argument('--phi_threshold', type=float, default=0.8, help="(Step 2) Phi coefficient threshold for gene co-occurrence (default: 0.8).")
    p_sa.add_argument('--p_value_threshold', type=float, default=0.01, help="(Step 2) P-value threshold for gene co-occurrence (default: 0.01).")
    p_sa.add_argument('--adjacency_distance', type=int, default=250, help="(Step 2) Max distance (bp) between adjacent genes in a module (default: 250).")
    p_sa.add_argument('--support_threshold', type=float, default=0.8, help="(Step 2) Min fraction of genomes where a module must be physically adjacent (default: 0.8).")
    # ============================= END OF S1 MODIFICATION =============================

    p_st = subparsers.add_parser('run_stats', help='(Step 3) Run statistical tests to identify hotspots/coldspots.')
    p_st.add_argument('--slot_analysis_dir', required=True, help="Path to the output directory from the analyze_slots step.")
    p_st.add_argument('--output_dir', required=True, help='Path to the output directory for this module.')
    p_st.add_argument('--permutations', type=int, default=10000, help="Number of permutations to run (default: 10000).")
    p_st.add_argument('--dry_run', action='store_true', help='Check inputs and logic without running heavy computations.')

    p_ma = subparsers.add_parser('explore_mechanisms', help='(Step 4) Prepare data and run downstream mechanism exploration.')
    p_ma.add_argument('--analyses', required=True, nargs='+', choices=['motif', 'evo_dynamics', 'adjacency_analysis', 'extract_layouts', 'extract_cds_layout'], help='Select one or more analyses to run.')
    p_ma.add_argument('--def_arch_dir', required=True, help="Path to the output directory from the define_architecture step.")
    p_ma.add_argument('--slot_analysis_dir', required=True, help="Path to the output directory from the analyze_slots step.")
    p_ma.add_argument('--stats_dir', required=True, help="Path to the output directory from the run_stats step.")
    p_ma.add_argument('--pharokka_dir', required=True, help='(Required for motif/evo_dynamics) Path to the Pharokka output directory.')
    p_ma.add_argument('--output_dir', required=True, help='Path to the output directory for this module.')
    p_ma.add_argument('--spi_enrichment_threshold', type=float, default=1.0, help="(For evo_dynamics) SPI value above which a module is 'high-SPI' (default: 1.0).")
    p_ma.add_argument('--dry_run', action='store_true', help='Check inputs and logic without running heavy computations.')

    # --- Argument Parsing and Dispatch ---\
    args = parser.parse_args()

    # Set faa_dir from pharokka_dir for compatibility
    if hasattr(args, 'pharokka_dir'):
        args.faa_dir = args.pharokka_dir
        
    if args.command == 'run':
        run_pipeline(args)
    elif args.command == 'define_architecture':
        run_da(args)
    elif args.command == 'analyze_slots':
        run_sa(args)
    elif args.command == 'run_stats':
        run_st(args)
    elif args.command == 'explore_mechanisms':
        run_ma(args)

if __name__ == '__main__':
    main()

## summarizer.py

In [ ]:
# /GenoSynax_test/genosyntax/summarizer.py

import json
import os
import pandas as pd
import numpy as np
from scipy.stats import entropy

def _load_data(path, file_type='json'):
    """Helper function to safely load JSON or CSV files."""
    if not os.path.exists(path):
        print(f"  [Info] Optional input file not found, will proceed without it: {os.path.basename(path)}")
        return None
    try:
        if file_type == 'json':
            with open(path, 'r') as f:
                return json.load(f)
        elif file_type == 'csv':
            return pd.read_csv(path)
    except Exception as e:
        print(f"  [ERROR] Could not load or parse file {path}: {e}")
        return None

def calculate_gri(arch_def, weights={'c': 0.6, 's': 0.3, 'm': 0.1}):
    """
    Calculates the Genomic Rigidity Index (GRI) based on the architecture definition.
    """
    if not arch_def or not arch_def.get("anchor_definitions"):
        return {
            "GRI_score": 0.0,
            "details": {
                "backbone_coverage": 0.0,
                "backbone_stability": 0.0,
                "macro_conservation": 0.0,
            },
            "status": "No anchor modules were found, rigidity is considered minimal."
        }

    # 1. Backbone Coverage (C_backbone)
    anchor_core_genes = {gene for anchor in arch_def['anchor_definitions'] for gene in anchor['skeleton']}
    total_core_genes = set(arch_def.get('metadata', {}).get('pangenome_stratification', {}).get('strict_core', []))
    c_backbone = len(anchor_core_genes) / len(total_core_genes) if total_core_genes else 0.0

    # 2. Backbone Stability (S_backbone)
    anchors = arch_def['anchor_definitions']
    avg_coverage = np.mean([a.get('coverage_fraction', 0) for a in anchors])
    avg_asi = np.mean([a.get('asi', 0) for a in anchors])
    avg_aoi = np.mean([a.get('aoi', 0) for a in anchors])
    s_backbone = (avg_coverage + avg_asi + avg_aoi) / 3 if anchors else 0.0

    # 3. Macro-structural Conservation (M_conservation)
    macro_arch = arch_def.get('macro_architectures', {})
    E = macro_arch.get('global_permutation_entropy', 0)
    N = macro_arch.get('total_architectures_found', 1)
    if N <= 1:
        m_conservation = 1.0
    else:
        E_max = np.log2(N)
        m_conservation = 1 - (E / E_max) if E_max > 0 else 1.0
        
    gri_score = (weights['c'] * c_backbone +
                 weights['s'] * s_backbone +
                 weights['m'] * m_conservation)

    return {
        "GRI_score": round(gri_score, 4),
        "details": {
            "backbone_coverage": round(c_backbone, 4),
            "backbone_stability": round(s_backbone, 4),
            "macro_conservation": round(m_conservation, 4),
        },
        "status": "Calculated successfully."
    }

def _calculate_hotspot_activity(stats_df):
    """Helper to calculate total activity score from a permutation test report."""
    if stats_df is None or stats_df.empty:
        return 0.0
    
    hotspots = stats_df[stats_df['p_value_enrichment'] < 0.05].copy()
    if hotspots.empty:
        return 0.0
        
    hotspots.loc[:, 'p_value_enrichment'] = hotspots['p_value_enrichment'].replace(0, 1e-300)
    hotspots.loc[:, 'fold_enrichment'] = pd.to_numeric(hotspots['fold_enrichment'], errors='coerce').fillna(0)
    
    hotspots['score'] = hotspots['fold_enrichment'] * (-np.log10(hotspots['p_value_enrichment']))
    return hotspots['score'].sum()

def calculate_gps(module_dynamics_report, stats_report_all, stats_report_multi, weights={'m': 0.4, 'h': 0.3, 'c': 0.3}):
    """
    Calculates the Genomic Plasticity Score (GPS).
    """
    # 1. Genetic Mobility (M_genetic)
    if module_dynamics_report is not None and not module_dynamics_report.empty:
        total_freq = module_dynamics_report['frequency'].sum()
        m_genetic = (module_dynamics_report['frequency'] * module_dynamics_report['spi']).sum() / total_freq if total_freq > 0 else 1.0
    else:
        m_genetic = 1.0

    # 2. Hotspot Activity (H_activity)
    h_activity = _calculate_hotspot_activity(stats_report_all)

    # 3. Cassette-Driven Plasticity (C_cassette)
    c_cassette = _calculate_hotspot_activity(stats_report_multi)

    gps_score = (weights['m'] * m_genetic +
                 weights['h'] * h_activity +
                 weights['c'] * c_cassette)
                 
    return {
        "GPS_score": round(gps_score, 4),
        "details": {
            "genetic_mobility": round(m_genetic, 4),
            "hotspot_activity_all_genes": round(h_activity, 4),
            "hotspot_activity_cassettes": round(c_cassette, 4),
        }
    }

def extract_key_findings(arch_def, module_dynamics_report, stats_report_all):
    """Extracts human-readable highlights from the analysis."""
    findings = {}
    
    # Architecture findings
    if arch_def and arch_def.get("anchor_definitions"):
        major_arch = arch_def.get('macro_architectures', {}).get('major_architecture', {})
        findings['dominant_architecture'] = {
            "permutation": " -> ".join(major_arch.get('permutation', [])),
            "frequency": major_arch.get('frequency', 0)
        }
        findings['anchor_module_count'] = len(arch_def["anchor_definitions"])
    else:
        findings['dominant_architecture'] = "No stable architecture found."
        findings['anchor_module_count'] = 0

    # Plasticity findings
    if module_dynamics_report is not None and not module_dynamics_report.empty:
        top_plastic = module_dynamics_report.sort_values(by='spi', ascending=False).head(5)
        findings['top_5_most_plastic_modules'] = top_plastic[['module_id', 'spi', 'frequency', 'consensus_function']].to_dict('records')
    
    if stats_report_all is not None and not stats_report_all.empty:
        top_hotspots = stats_report_all[stats_report_all['p_value_enrichment'] < 0.05].sort_values(by='p_value_enrichment').head(3)
        findings['top_3_insertion_hotspots'] = top_hotspots[['slot_type', 'p_value_enrichment', 'fold_enrichment']].to_dict('records')
        
    return findings


def run(main_output_dir):
    """
    Entrypoint function to generate the comprehensive summary report.
    """
    print("\n--- Running Step 3.5: Generating Comprehensive Summary Report ---")
    
    # Define paths
    arch_def_path = os.path.join(main_output_dir, "1_define_architecture", "architecture_definition.json")
    module_report_path = os.path.join(main_output_dir, "2_slot_analysis", "reports", "module_dynamics_report.csv")
    stats_all_path = os.path.join(main_output_dir, "3_statistical_test", "reports", "permutation_test_slot_significance.csv")
    stats_multi_path = os.path.join(main_output_dir, "3_statistical_test", "reports", "permutation_test_multigene_module_significance.csv")
    
    # Load data
    arch_def = _load_data(arch_def_path, 'json')
    module_dynamics_report = _load_data(module_report_path, 'csv')
    stats_report_all = _load_data(stats_all_path, 'csv')
    stats_report_multi = _load_data(stats_multi_path, 'csv')

    if not arch_def:
        print("  [FATAL] `architecture_definition.json` is essential and was not found. Cannot generate summary.")
        return False

    # Perform calculations
    gri_results = calculate_gri(arch_def)
    gps_results = calculate_gps(module_dynamics_report, stats_report_all, stats_report_multi)
    key_findings = extract_key_findings(arch_def, module_dynamics_report, stats_report_all)

    # Assemble final report
    summary_report = {
        "metadata": {
            "report_generator": "GenoSyntax v1.0 Summarizer",
            "qc_summary": arch_def.get("metadata", {}).get("qc_summary", {}),
            "analysis_parameters": arch_def.get("metadata", {}).get("analysis_parameters", {}),
        },
        "genomic_rigidity_index": gri_results,
        "genomic_plasticity_profile": {
            "genetic_mobility_score": gps_results["details"]["genetic_mobility"],
            "general_insertion_activity_score": gps_results["details"]["hotspot_activity_all_genes"],
            "cassette_driven_plasticity_score": gps_results["details"]["hotspot_activity_cassettes"],
            "legacy_gps_combined_score": gps_results["GPS_score"]
        },
        "key_findings_summary": key_findings
    }

    # Save the report
    report_path = os.path.join(main_output_dir, "comprehensive_summary_report.json")
    try:
        with open(report_path, 'w') as f:
            json.dump(summary_report, f, indent=4)
        print(f"[*] Comprehensive summary report saved successfully to: {report_path}")
    except IOError as e:
        print(f"  [FATAL] Could not write summary report to {report_path}: {e}")
        return False
        
    return True

## test_main_script.sh

In [ ]:
#!/bin/bash

# ==============================================================================
# GenoSyntax Dry Run Test Script
#
# This script tests the new 'run' command and its flow control parameters
# using the --dry_run flag. It creates a dummy file structure to satisfy
# the input path checks.
# ==============================================================================

echo "====== Setting up dummy file structure for dry run tests ======"

# Create a temporary directory for all test inputs and outputs
TEST_DIR="genosyntax_test_environment"
rm -rf ${TEST_DIR}
mkdir -p ${TEST_DIR}

# --- Create dummy input directories and files ---
mkdir -p ${TEST_DIR}/inputs/panaroo
mkdir -p ${TEST_DIR}/inputs/pharokka
mkdir -p ${TEST_DIR}/inputs/fastas
mkdir -p ${TEST_DIR}/inputs/faas
mkdir -p ${TEST_DIR}/inputs/phenotypes

touch ${TEST_DIR}/inputs/panaroo/gene_presence_absence_roary.csv
touch ${TEST_DIR}/inputs/phenotypes/pheno.csv

# --- Create a main output directory ---
mkdir -p ${TEST_DIR}/pipeline_output

echo "Dummy structure created in '${TEST_DIR}'."
echo ""


# ==============================================================================
# --- TEST CASES ---
# ==============================================================================

echo "====== Running Test Cases for the 'run' command with --dry_run ======"
echo "NOTE: These are dry runs. No actual computation will be performed."
echo "----------------------------------------------------------------------"

# --- Test Case 1: Default behavior ---
# Should run steps 1, 2, and 3.
echo ">>> Test Case 1: Default run (stops after step 3)"
python main.py run \
    --panaroo_file ${TEST_DIR}/inputs/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir ${TEST_DIR}/inputs/pharokka \
    --fasta_dir ${TEST_DIR}/inputs/fastas \
    --faa_dir ${TEST_DIR}/inputs/faas \
    --output_dir ${TEST_DIR}/pipeline_output/run1 \
    --dry_run

echo "----------------------------------------------------------------------"

# --- Test Case 2: Stop after Step 1 ---
echo ">>> Test Case 2: Run with --until-step 1"
python main.py run \
    --panaroo_file ${TEST_DIR}/inputs/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir ${TEST_DIR}/inputs/pharokka \
    --fasta_dir ${TEST_DIR}/inputs/fastas \
    --faa_dir ${TEST_DIR}/inputs/faas \
    --output_dir ${TEST_DIR}/pipeline_output/run2 \
    --until-step 1 \
    --dry_run

echo "----------------------------------------------------------------------"

# --- Test Case 3: Run all steps, including selected mechanisms ---
# Should run steps 1, 2, 3, and 4 (motif and gwas).
echo ">>> Test Case 3: Run all steps with selected mechanisms (--mechanisms motif gwas)"
python main.py run \
    --panaroo_file ${TEST_DIR}/inputs/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir ${TEST_DIR}/inputs/pharokka \
    --fasta_dir ${TEST_DIR}/inputs/fastas \
    --faa_dir ${TEST_DIR}/inputs/faas \
    --phenotype_file ${TEST_DIR}/inputs/phenotypes/pheno.csv \
    --output_dir ${TEST_DIR}/pipeline_output/run3 \
    --mechanisms motif gwas \
    --dry_run

echo "----------------------------------------------------------------------"

# --- Test Case 4: Run all 4 steps implicitly ---
# --until-step 4 should trigger step 4 even without the --mechanisms flag.
echo ">>> Test Case 4: Run all steps implicitly with --until-step 4"
python main.py run \
    --panaroo_file ${TEST_DIR}/inputs/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir ${TEST_DIR}/inputs/pharokka \
    --fasta_dir ${TEST_DIR}/inputs/fastas \
    --faa_dir ${TEST_DIR}/inputs/faas \
    --phenotype_file ${TEST_DIR}/inputs/phenotypes/pheno.csv \
    --output_dir ${TEST_DIR}/pipeline_output/run4 \
    --until-step 4 \
    --dry_run
    
echo "----------------------------------------------------------------------"
echo "====== All dry run tests complete. ======"
echo "Please review the output above to ensure each step was called as expected."
echo "You can now clean up the test environment by running: rm -rf ${TEST_DIR}"

genosyntax/utils/file_ops.py

In [ ]:
# /GenoSynax_test/genosyntax/utils/file_ops.py

import json
import os
import subprocess
from itertools import combinations
import numpy as np
from scipy.stats import spearmanr
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

def load_json(file_path):
    """Safely loads a JSON file."""
    try:
        with open(file_path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[ERROR] JSON file not found: {file_path}")
        return None

def save_json(data, file_path):
    """Saves data to a JSON file with indentation."""
    try:
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=4)
        return True
    except IOError as e:
        print(f"[ERROR] Could not write to JSON file {file_path}: {e}")
        return False

def calculate_asi_for_gene(sequences, temp_dir):
    """
    Calculates the average pairwise identity for a single gene's sequences
    using an external alignment tool like MAFFT.
    """
    if len(sequences) < 2:
        return 1.0

    pid = os.getpid()
    temp_fasta_path = os.path.join(temp_dir, f"temp_asi_{pid}.fasta")
    
    records = [SeqRecord(Seq(seq), id=f"seq_{i}") for i, seq in enumerate(sequences)]
    SeqIO.write(records, temp_fasta_path, "fasta")

    aln_path = temp_fasta_path + ".aln"
    
    try:
        subprocess.run(
            ["mafft", "--auto", "--quiet", temp_fasta_path],
            stdout=open(aln_path, "w"),
            stderr=subprocess.DEVNULL,
            text=True,
            check=True
        )
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        if isinstance(e, FileNotFoundError):
            print("\n[FATAL ERROR] MAFFT command not found. Is it installed and in your PATH?")
            print("  To install: conda install -c bioconda mafft")
            exit(1)
        return None

    try:
        alignment = list(SeqIO.parse(aln_path, "fasta"))
    except Exception:
        alignment = []
        
    pids = []
    if alignment:
        for rec1, rec2 in combinations(alignment, 2):
            seq1_str, seq2_str = str(rec1.seq), str(rec2.seq)
            matches = sum(1 for a, b in zip(seq1_str, seq2_str) if a == b and a != '-')
            len1 = len(seq1_str.replace('-', ''))
            len2 = len(seq2_str.replace('-', ''))
            min_len = min(len1, len2)
            if min_len > 0:
                pids.append(matches / min_len)
    
    if os.path.exists(temp_fasta_path): os.remove(temp_fasta_path)
    if os.path.exists(aln_path): os.remove(aln_path)

    return np.mean(pids) if pids else 1.0

# --- MODIFIED FUNCTION ---
def calculate_gc_content(sequence):
    """
    Calculates GC content of a DNA sequence string.
    This is now the primary function for boundary signal analysis.
    """
    if not isinstance(sequence, str) or len(sequence) == 0:
        return 0.0
    g = sequence.upper().count('G')
    c = sequence.upper().count('C')
    return (g + c) / len(sequence)

/genosyntax/visualization/plot_genome_layouts.py

In [ ]:
# --- FINAL, STANDALONE PLOTTING SCRIPT (with consistent coloring) ---
# To be placed in: /genosyntax/visualization/plot_genome_layouts.py

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import argparse

# A predefined list of visually distinct, colorblind-friendly colors.
# This ensures that the same anchor ID gets the same color across different plots.
PREDEFINED_COLORS = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
    '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5'
]

def plot_layouts(input_csv, output_image, target_genomes_file=None, sort_by_structure=False):
    """
    Generates a comparative visualization of genome layouts using the CSV
    output from the GenoSyntax 'extract_layouts' analysis.
    """
    print("--- GenoSyntax Layout Plotter ---")
    
    # --- 1. Load and prepare data ---
    print(f"[*] Reading layout data from: {input_csv}")
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"\\n[FATAL ERROR] Input file not found: {input_csv}")
        print("Please run the 'extract_layouts' analysis in GenoSyntax first to generate it.")
        return

    if df.empty:
        print("[INFO] The layout data file is empty. Nothing to plot.")
        return

    # --- 2. Determine the order of genomes to plot ---
    genome_order = []
    if target_genomes_file:
        print(f"[*] Reading target genome order from: {target_genomes_file}")
        try:
            with open(target_genomes_file, 'r') as f:
                genome_order = [line.strip() for line in f if line.strip()]
            
            # Filter the dataframe to only include target genomes
            df = df[df['genome_id'].isin(genome_order)].copy()
            # Enforce the specified order
            df['genome_id'] = pd.Categorical(df['genome_id'], categories=genome_order, ordered=True)
            df = df.sort_values('genome_id')
        except FileNotFoundError:
            print(f"  [WARNING] Target genomes file not found. Plotting all genomes.")
            genome_order = sorted(df['genome_id'].unique())
    elif sort_by_structure:
        print("[*] Sorting all genomes by their structural signature for clarity...")
        signatures = df.groupby('genome_id')['element_id'].apply(lambda x: ''.join(x)).reset_index(name='signature')
        genome_order = signatures.sort_values('signature')['genome_id'].tolist()
        df['genome_id'] = pd.Categorical(df['genome_id'], categories=genome_order, ordered=True)
        df = df.sort_values('genome_id')
    else:
        print("[*] Plotting all genomes in default order...")
        genome_order = sorted(df['genome_id'].unique())

    if not genome_order:
        print("[ERROR] No genomes to plot after filtering. Please check your target genomes file.")
        return

    # --- 3. Prepare for plotting (colors, figure) ---
    print(f"[*] Preparing to plot {len(genome_order)} genomes...")
    
    # Create a consistent color map for anchors
    anchor_ids = sorted([eid for eid in df['element_id'].unique() if eid.startswith('Anchor_')], 
                        key=lambda x: int(x.split('_')[1]))
    
    anchor_color_map = {}
    for anchor_id in anchor_ids:
        try:
            # Use the anchor's number to get a consistent color index
            anchor_num = int(anchor_id.split('_')[1])
            color_index = (anchor_num - 1) % len(PREDEFINED_COLORS)
            anchor_color_map[anchor_id] = PREDEFINED_COLORS[color_index]
        except (ValueError, IndexError):
            # Fallback for unusually named anchors
            anchor_color_map[anchor_id] = 'grey'

    # Dynamically size the figure based on the number of genomes
    fig_height = max(5, len(genome_order) * 0.4 + 1)
    fig, ax = plt.subplots(figsize=(16, fig_height))
    
    max_coord = df['end'].max() if not df.empty else 1

    # --- 4. The Core Plotting Loop ---
    for i, genome_id in enumerate(genome_order):
        y_pos = i
        genome_df = df[df['genome_id'] == genome_id]
        
        for _, row in genome_df.iterrows():
            element_type = row['element_type']
            element_id = row['element_id']
            start = row['start']
            width = row['length']
            
            # Determine color and height
            if element_type == 'Anchor':
                color = anchor_color_map.get(element_id, 'grey')
                height = 0.8
                y_offset = -0.4
            else: # Slot
                color = '#E5E5E5' # A light grey for all slots
                height = 0.7 # Make slots slightly thinner
                y_offset = -0.35

            rect = mpatches.Rectangle((start, y_pos + y_offset), width, height, 
                                     linewidth=0.5, edgecolor='white', facecolor=color, alpha=0.9)
            ax.add_patch(rect)

    # --- 5. Formatting the plot ---
    ax.set_yticks(range(len(genome_order)))
    ax.set_yticklabels(genome_order, fontsize=10)
    ax.set_ylim(-0.5, len(genome_order) - 0.5)
    
    ax.set_xlabel("Genomic Coordinate (bp)", fontsize=14)
    ax.set_xlim(0, max_coord * 1.05)
    
    # Create a sorted legend
    legend_patches = []
    for anchor_id in anchor_ids:
        legend_patches.append(mpatches.Patch(color=anchor_color_map[anchor_id], label=anchor_id))
    legend_patches.append(mpatches.Patch(color='#E5E5E5', label='Slot'))
    
    ax.legend(handles=legend_patches, bbox_to_anchor=(1.01, 1), loc='upper left', title="Element Key", fontsize=12)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis() # Puts the first genome at the top

    # Add a clean background and grid
    ax.set_facecolor('white')
    ax.grid(axis='x', color='grey', linestyle=':', linewidth=0.5)
    ax.set_axisbelow(True) # Ensure grid is behind the bars

    plt.tight_layout(rect=[0, 0, 0.88, 1]) # Adjust layout to make space for the legend

    # --- 6. Save the final image ---
    try:
        plt.savefig(output_image, dpi=300, bbox_inches='tight')
        print(f"\\n[SUCCESS] Genome layout plot saved successfully!")
        print(f"[*] Output file is: {output_image}")
    except Exception as e:
        print(f"\\n[ERROR] Failed to save the plot: {e}")
    finally:
        plt.close(fig)

# NOTE: The __main__ block is removed from this file.
# The command-line interface will now be handled by main.py.

MANUAL.md


# GenoSyntax v1.0 - User Manual

## 1. Introduction

### 1.1. Overview
GenoSyntax is a comprehensive bioinformatics framework for the analysis of genomic architecture and plasticity in related microbial genomes, particularly suited for phage and bacterial pangenomes. It deconstructs complex pangenomes into a stable, conserved **backbone** (defined by "Anchor Modules") and dynamic **slots** where mobile genetic elements reside.

The framework is designed as a sequential pipeline that automates the entire analytical process, from raw annotations to quantitative, high-level insights. It culminates in the generation of two novel metrics—the **Genomic Rigidity Index (GRI)** and the **Genomic Plasticity Score (GPS)**—which allow for direct, quantitative comparison of architectural dynamics between different datasets.

### 1.2. Conceptual Workflow
The analysis is performed in four main stages:
1.  **Define Architecture:** Identify the conserved genomic backbone.
2.  **Analyze Slots:** Characterize the plastic regions and the mobile elements within them.
3.  **Statistical Testing:** Determine the significance of mobile element insertion patterns.
4.  **Explore Mechanisms:** Prepare data to investigate the molecular and evolutionary drivers of the observed architecture.

---

## 2. Installation

GenoSyntax requires a Conda environment to manage its dependencies and external tools.

### 2.1. Prerequisites
-   **Conda:** [Miniconda](https://docs.conda.io/en/latest/miniconda.html) or Anaconda must be installed.
-   **Git:** For cloning the repository.

### 2.2. Installation Steps

1.  **Clone the Repository:**
    ```bash
    git clone https://github.com/yunzhe-liu/GenoSynax.git
    cd GenoSyntax
    ```

2.  **Create and Activate Conda Environment:**
    A dedicated environment is highly recommended.
    ```bash
    conda create -n genosyntax python=3.9
    conda activate genosyntax
    ```

3.  **Install Dependencies:**
    Install the required Python packages and external bioinformatics tools.
    ```bash
    # Install Python packages from the requirements file
    pip install -r requirements.txt
    
    # Install essential external tools via Conda
    conda install -c bioconda mafft
    ```

---

## 3. Input Data Requirements

GenoSyntax requires four sources of data, provided as command-line arguments.

| Argument                | Description                                                                                                                                                             | Source Tool |
| ----------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------- |
| `--panaroo_file`        | The `gene_presence_absence_roary.csv` file. This provides the pangenome definition and gene family prevalence.                                                           | Panaroo     |
| `--pharokka_dir`        | The main output directory from Pharokka. GenoSyntax expects this directory to contain a subdirectory for each genome, which holds the `.gff`, `.faa`, and reoriented `.fna` files. | Pharokka    |
| `--fasta_dir`           | A directory containing the original, raw FASTA sequence files for each genome (one file per genome). This is used for definitive assembly quality control.                  | User        |
| `--faa_dir`     | A directory containing protein FASTA (`.faa`) files. For Pharokka users, this is the same path as `--pharokka_dir`.                                                        | Pharokka    |

---

## 4. The GenoSyntax Pipeline

### 4.1. Recommended Usage: The `run` Command

For most users, the integrated `run` command is the recommended entrypoint. It automates the entire workflow, manages all intermediate files, and provides a single, unified interface for all parameters.

**Default Usage (Core Analysis):**
This command executes Steps 1, 2, and 3, culminating in the statistical analysis and the generation of the `comprehensive_summary_report.json`. This is sufficient for most standard analyses.

```bash
python main.py run \
    --panaroo_file /path/to/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --fasta_dir /path/to/raw_genome_fastas/ \
    --output_dir ./My_GenoSyntax_Analysis/
```

**Full Pipeline with Mechanism Exploration:**
To run all four steps, including the final mechanism analysis, use the `--mechanisms` flag and set `--until-step 4`.

```bash
python main.py run \
    --panaroo_file /path/to/panaroo/gene_presence_absence_roary.csv \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --fasta_dir /path/to/raw_genome_fastas/ \
    --output_dir ./My_Full_Analysis/ \
    --mechanisms motif evo_dynamics adjacency_analysis extract_layouts \
    --until-step 4
```

### 4.2. Detailed Module Breakdown

#### Module 1: `define_architecture`

**Purpose:** This foundational module identifies and characterizes the stable structural backbone of the genome cohort.

**Key Parameters:**
*   `--max_contigs` (Default: `1`): The maximum number of contigs a genome can have to pass assembly QC.
*   `--tolerance` (Default: `2`): The maximum number of accessory genes allowed between two core genes within an anchor.
*   `--coverage_threshold` (Default: `0.9`): The minimum fraction of high-quality genomes in which an anchor must be found.
*   `--min_length` (Default: `2`): The minimum number of core genes required to form a valid anchor.
*   `--strict_core_threshold` (Default: `1.0`): Prevalence threshold for a gene to be 'strict core'.
*   `--soft_core_threshold` (Default: `0.98`): Prevalence threshold for a gene to be 'soft core', also defines the boundary for accessory genes.

**Key Outputs:**
*   `1_define_architecture/architecture_definition.json`: The master blueprint of the genomic architecture. It contains:
    *   `metadata`: QC summary and all analysis parameters.
    *   `anchor_definitions`: A detailed list of each anchor, including its gene skeleton, coverage, sequence conservation (**ASI**), structural rigidity (**AOI**), and functional class.
    *   `macro_architectures`: A summary of the global arrangement of anchors, including the dominant permutation and overall structural entropy.

#### Module 2: `analyze_slots`

**Purpose:** This module analyzes the plastic regions of the genome (the "slot space"), discovers modular accessory elements, and quantifies their structural dynamics.

**Key Parameters:**
*   `--spi_resolution` (Default: `'low'`): Sets the granularity for SPI calculation. `'low'` resolution is recommended as it aggregates related slots and reduces noise. `'high'` resolution treats every unique inter-core-gene position as distinct.
*   `--phi_threshold` (Default: `0.8`): Phi coefficient threshold for identifying statistical co-occurrence between accessory genes.
*   `--p_value_threshold` (Default: `0.01`): P-value threshold for gene co-occurrence.
*   `--adjacency_distance` (Default: `250`): Maximum distance (in bp) between adjacent genes to be considered part of the same physical module.
*   `--support_threshold` (Default: `0.8`): Minimum fraction of genomes where a candidate module's members must be physically adjacent to be validated.

**Key Outputs:**
*   `2_slot_analysis/reports/module_dynamics_report.csv`: The central report for plasticity analysis. Each row represents an Accessory Module (AM), with key columns:
    *   `spi`: The **Structural Plasticity Index**, indicating how many different structural locations the module occupies.
    *   `frequency`: The prevalence of the module in the population.
    *   `behavioral_class`: A classification based on SPI and frequency (e.g., "Multi-Position", "Structurally Conserved").

#### Module 3: `run_stats`

**Purpose:** To apply rigorous statistical testing to determine if the observed distribution of accessory modules is non-random, thereby identifying insertion "hotspots" and "coldspots".

**Key Parameters:**
*   `--permutations` (Default: `10000`): The number of simulations to perform for the permutation test. Higher numbers increase precision but take longer.

**Key Outputs:**
This module performs a **dual-tier analysis** and generates two key reports:
1.  `3_statistical_test/reports/permutation_test_slot_significance.csv`: The primary report testing for hotspots/coldspots based on the insertion of **any accessory gene**. Use this to understand broad patterns of genomic plasticity.
2.  `3_statistical_test/reports/permutation_test_multigene_module_significance.csv`: An additional report testing specifically for hotspots/coldspots of **multi-gene modules (cassettes)**. This is highly recommended for discovering hotspots likely governed by dedicated insertion mechanisms (e.g., site-specific integrases).

#### Module 4: `explore_mechanisms`

**Purpose:** The final analytical stage, designed to prepare high-quality data for specialized downstream tools to explore the mechanisms underlying the observed architectural patterns.

**Key Parameters:**
*   `--analyses`: A space-separated list of which analyses to run. Choices:
    *   `motif`: Prepare FASTA files of slot boundaries for motif discovery.
    *   `evo_dynamics`: Analyze the relationship between module mobility (SPI) and protein evolutionary rate.
    *   `adjacency_analysis`: Perform a deep functional analysis of multi-gene modules.
    *   `extract_layouts`: Generate a simple coordinate map of all anchors and slots for visualization.
    *   `extract_cds_layout`: Generate a detailed table of every Coding Sequence (CDS) in all high-quality genomes, including coordinates and functional category (CDS layout).
*   `--spi_enrichment_threshold` (Default: `1.0`): For `evo_dynamics`, the SPI value above which a module is considered "highly plastic".

**Key Outputs:**
*   **For `motif` analysis:** Creates two directories:
    *   `4_mechanism_analysis/motif_analysis_all_modules/`: FASTA files of boundaries from general hotspots.
    *   `4_mechanism_analysis/motif_analysis_multigene_modules/`: FASTA files from cassette-specific hotspots (recommended for finding integration motifs).
    *   `4_mechanism_analysis/motif_analysis_high_spi_enriched/`: Contains boundary sequences from slots that are statistically enriched for hosting **highly mobile (high-SPI) modules**. This dataset is ideal for exploratory analysis aimed at identifying signals that might facilitate promiscuous or hyper-mobile element integration. *Note: This output is only generated if the `evo_dynamics` analysis is also run and identifies significant enrichment.*
*   **For `evo_dynamics` analysis:**
    *   `reports/spi_vs_ped_analysis.csv`: A report linking each module's structural plasticity (SPI) to its Protein Evolutionary Distance (PED).
    *   `reports/high_spi_slot_enrichment.csv`: A statistical report identifying slots preferentially targeted by highly plastic modules.
*   **For `adjacency_analysis` analysis:**
    *   `adjacency_module_analysis/`: A directory containing a narrative report, a CSV summary, and network files (`.csv`) for visualizing multi-gene module composition in tools like Cytoscape.
*   **For `extract_layouts` analysis:**
    *   `reports/all_genome_layouts.csv`: A coordinate map of all anchors and slots in every high-quality genome, perfect for input into plotting libraries like `ggplot2`'s `geom_rect`.
*   **For `extract_cds_layout` analysis:**
    *   `reports/all_genomes_cds_layout.csv`: A comprehensive table listing every CDS from all high-quality genomes, with columns for start/end coordinates, strand, product annotation, and a high-level functional category. This file is ideal for detailed gene-level visualizations and custom downstream analyses.


---

## 5. The Comprehensive Summary Report

After the core analysis (Steps 1-3) is complete, GenoSyntax generates a final, high-level summary file named `comprehensive_summary_report.json` in the main output directory. This report provides a quantitative snapshot of the cohort's architectural properties.

### 5.1. Genomic Rigidity Index (GRI)

The GRI is a composite score between **0 (highly fluid) and 1 (highly rigid)** that quantifies the stability of the genomic backbone.

-   **Interpretation:** A score near **1.0** indicates a highly conserved architecture where most core genes are locked into stable anchor modules with a consistent global arrangement. A score near **0** indicates a fluid pangenome with few or no identifiable anchors.
-   **Components:** The GRI is a weighted average of `backbone_coverage`, `backbone_stability`, and `macro_conservation`.

### 5.2. Genomic Plasticity Profile

Instead of a single, weighted score, GenoSyntax provides a **Genomic Plasticity Profile** consisting of three independent metrics. This allows for a more nuanced, multi-dimensional understanding of how plasticity is expressed in a given cohort. Comparing these scores between different datasets can reveal distinct evolutionary strategies.

-   **`genetic_mobility_score` (M_genetic):**
    *   **What it measures:** The population-weighted average mobility (SPI) of all accessory modules.
    *   **Interpretation:** A high score indicates that accessory modules in the population are frequently found in many different genomic locations. This points to a "promiscuous" or highly dynamic accessory genome where functional units are readily exchanged and moved.

-   **`general_insertion_activity_score` (H_activity):**
    *   **What it measures:** The cumulative statistical significance of all insertion hotspots for **any** type of accessory gene (single or multi-gene).
    *   **Interpretation:** A high score signifies that gene insertion is highly non-random and is strongly concentrated in specific genomic regions. This indicates the presence of powerful, broadly targeted insertion mechanisms or selection pressures favoring insertions at these loci.

-   **`cassette_driven_plasticity_score` (C_cassette):**
    *   **What it measures:** The cumulative statistical significance of insertion hotspots specifically for validated **multi-gene modules (cassettes)**.
    *   **Interpretation:** This is a highly specific metric. A high score strongly suggests the activity of dedicated molecular machinery (e.g., site-specific integrases) that recognizes and inserts organized, co-evolving functional cassettes. It provides powerful evidence for structured, mechanism-driven plasticity.

### 5.3. Key Findings Summary
This section provides a quick, human-readable overview of the most important results, including the dominant architecture, the top 5 most plastic modules, and the top 3 insertion hotspots.

---

### **6. Advanced Usage: A Guide to Step-by-Step Execution and Custom Workflows**

While the integrated `run` command is recommended for standard analyses, GenoSyntax's true power lies in its modular design. Executing each module as a separate command provides granular control, making it an ideal approach for debugging, parameter tuning, and creating custom analytical workflows.

In this mode, you are responsible for managing the input and output paths between steps. The fundamental rule is that the output of one step serves as the primary input for the next.

**General Help:**
You can get a detailed list of parameters for any command using the `-h` or `--help` flag.
```bash
# General help for all commands
python main.py -h

# Specific help for the define_architecture module
python main.py define_architecture -h
```

---

#### **Step 1: `define_architecture`**

**Purpose:** The foundational step that identifies the conserved structural backbone and produces the master `architecture_definition.json` blueprint.

**Command Template:**
```bash
python main.py define_architecture \
    --panaroo_file /path/to/gene_presence_absence_roary.csv \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --fasta_dir /path/to/raw_fastas/ \
    --output_dir ./output/1_define_architecture/ \
    --coverage_threshold 0.9 \
    --tolerance 2
```

*   **Key Inputs:** Raw data files (`--panaroo_file`, `--pharokka_dir`, `--fasta_dir`).
*   **Key Outputs:**
    *   `./output/1_define_architecture/architecture_definition.json` (The master blueprint).
    *   `./output/1_define_architecture/intermediate/` (Directory containing pre-processed databases like `gene_order_database.json`, which are essential for the next step).

*   **Advanced Use Case:** This module is ideal for iterative parameter tuning. If the initial run with default settings yields no anchors, you can re-run *only this step* with a lower `--coverage_threshold` (e.g., 0.8) to test for a less conserved, "dominant sub-structure" backbone without having to repeat any downstream analysis.

---

#### **Step 2: `analyze_slots`**

**Purpose:** To dissect the "plastic" regions of the genome, discover mobile accessory modules (AMs), and quantify their mobility (SPI).

**Command Template:**
```bash
python main.py analyze_slots \
    --architecture_file ./output/1_define_architecture/architecture_definition.json \
    --intermediate_dir_prev ./output/1_define_architecture/intermediate/ \
    --output_dir ./output/2_slot_analysis/ \
    --spi_resolution low
```

*   **Key Inputs:** The `architecture_definition.json` file and the `intermediate/` directory from the `define_architecture` step.
*   **Key Outputs:**
    *   `./output/2_slot_analysis/reports/module_dynamics_report.csv` (The central plasticity report).
    *   `./output/2_slot_analysis/intermediate/` (Contains the `slot_database.json` and `accessory_module_definitions.json`).

*   **Advanced Use Case:** You can experiment with different SPI calculation granularities. Run this step once with `--spi_resolution low` (default) for a broad overview, then re-run it with `--spi_resolution high` to investigate fine-grained positional shifts of specific modules.

---

#### **Step 3: `run_stats`**

**Purpose:** To apply rigorous permutation testing to determine if the observed distribution of AMs is statistically significant.

**Command Template:**
```bash
python main.py run_stats \
    --slot_analysis_dir ./output/2_slot_analysis/ \
    --output_dir ./output/3_statistical_test/ \
    --permutations 10000
```

*   **Key Inputs:** The entire output directory from the `analyze_slots` step.
*   **Key Outputs:**
    *   `./output/3_statistical_test/reports/permutation_test_slot_significance.csv`
    *   `./output/3_statistical_test/reports/permutation_test_multigene_module_significance.csv`

*   **Advanced Use Case:** For a final, publication-quality analysis, you might want to increase the number of simulations for higher confidence in your p-values. You can easily re-run just this step with `--permutations 100000` after completing the initial, faster exploration.

---

#### **Step 4: `explore_mechanisms`**

**Purpose:** To synthesize all prior findings and prepare high-quality data for specialized downstream tools (MEME, Cytoscape, etc.).

**Command Template:**
```bash
python main.py explore_mechanisms \
    --analyses motif evo_dynamics extract_layouts \
    --def_arch_dir ./output/1_define_architecture/ \
    --slot_analysis_dir ./output/2_slot_analysis/ \
    --stats_dir ./output/3_statistical_test/ \
    --pharokka_dir /path/to/pharokka_outputs/ \
    --output_dir ./output/4_mechanism_analysis/
```

*   **Key Inputs:** The output directories from all three previous steps are required, as this module integrates data from the entire pipeline. The raw `pharokka_dir` is also needed for sequence extraction.
*   **Key Outputs:** A series of directories and files within `./output/4_mechanism_analysis/` tailored to the requested `--analyses` (e.g., `motif_analysis_multigene_modules/`, `reports/all_genome_layouts.csv`).

*   **Advanced Use Case:** This module is designed for selective execution. After a full analysis, you might realize you also want to analyze adjacency modules. You can simply run `python main.py explore_mechanisms --analyses adjacency_analysis ...` without having to re-compute anything from steps 1-3.

---

## 7. Troubleshooting

-   **`mafft: command not found`**: This is the most common error. It means the `mafft` executable is not in your system's PATH. Ensure you have activated the correct conda environment (`conda activate genosyntax`) where `mafft` was installed.
-   **File Not Found Errors**: Double-check all input paths. The `run` command's `--dry_run` flag is excellent for validating paths before starting a long computation.
-   **No Anchors Found**: If the `define_architecture` step completes but finds no anchors, this is a valid biological result indicating a highly fluid genome cohort. The pipeline will gracefully skip downstream steps that depend on anchors. Consider re-running with a lower `--coverage-threshold` (e.g., 0.7 or 0.6) to search for less-conserved "dominant sub-structure backbones."
```

In [ ]:

python main.py explore_mechanisms \
    --analyses motif\
    --def_arch_dir ./Final_Analysis/1_define_architecture/ \
    --slot_analysis_dir ./Final_Analysis/2_slot_analysis/ \
    --stats_dir ./Final_Analysis/3_statistical_test/ \
    --pharokka_dir /cluster/home/yunzliu/Project/Multi-Cluster/cluster_Janus/pharokka_output \
    --output_dir ./Final_Analysis/4_mechanism_analysis/


python main.py run \
    --panaroo_file /cluster/home/yunzliu/Project/Multi-Cluster/cluster_Janus/panaroo_analysis/panaroo_output_strict/gene_presence_absence_roary.csv \
    --pharokka_dir /cluster/home/yunzliu/Project/Multi-Cluster/cluster_Janus/pharokka_output \
    --fasta_dir /cluster/home/yunzliu/Project/Multi-Cluster/cluster_Janus/pharokka_input/ \
    --output_dir ./Test/ \
    --mechanisms motif evo_dynamics adjacency_analysis extract_layouts extract_cds_layout \
    --until-step 4